# Train everything on a free Colab T4, once

Colab is only the GPU; the deliverable is the GitHub repo. This notebook trains all models, runs the experiments and packs the
results into one zip that you unzip into your local repo and commit.

1. `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
2. Everything is stored in your **Google Drive** (`MyDrive/GPT-2`). If Colab disconnects, open the notebook again and
   **Run all**: finished trainings are skipped and interrupted ones resume from their last checkpoint.
3. Total time is roughly 2.5-3 hours of GPU (my estimate: ~30 min per main model, less for the sweep). Free Colab may need two sessions.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # should say Tesla T4

## 1. Code (kept in Google Drive so nothing is lost on disconnect)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
REPO = "/content/drive/MyDrive/GPT-2"
if not os.path.exists(REPO):
    !git clone https://github.com/Efrata-Habte/GPT-2.git {REPO}
else:   # refresh the code to the latest version on GitHub; trained models and results in Drive are left alone
    !git -C {REPO} fetch -q origin && git -C {REPO} checkout -q origin/main -- src data experiments scripts configs app
%cd {REPO}
!pip install -q -r requirements.txt

## 2. Data (downloads once, a couple of minutes)

In [ ]:
!python data/prepare.py                       # Tiny Shakespeare            (small English data)
!python -I data/amharic/prepare.py            # MasakhaNEWS Amharic news    (small Amharic data)
!python -I data/prepare_full_shakespeare.py   # Complete Works of Shakespeare (full English data) + shared test set
!python -I data/amharic/prepare_wikipedia.py  # Amharic Wikipedia + news    (full Amharic data) + shared test sets

## 3. Train the four models
`train()` skips a finished run and resumes an interrupted one.

In [ ]:
def train(config, out_dir):
    if os.path.exists(f"{out_dir}/summary.json"):
        print(f"{out_dir}: already trained, skipping")
        return
    resume = f" --resume {out_dir}/ckpt_last.pt" if os.path.exists(f"{out_dir}/ckpt_last.pt") else ""
    get_ipython().system(f"python -m src.train --config {config}{resume}")

In [ ]:
train("configs/small.yaml", "checkpoints/small")                 # the assignment model: English, Tiny Shakespeare

In [ ]:
train("configs/amharic.yaml", "checkpoints/amharic")             # Amharic, news

In [ ]:
train("configs/small_full.yaml", "checkpoints/small_full")       # English, Complete Works (more data)

In [ ]:
train("configs/amharic_full.yaml", "checkpoints/amharic_full")   # Amharic, Wikipedia + news (more data)

## 4. Experiment 4: architecture sweep (3 shorter runs; finished runs are reused)

In [ ]:
!python experiments/exp4_architecture.py --param n_layer

## 5. Experiments that need no extra training (minutes)

In [ ]:
!python experiments/exp1_temperature.py          # temperature vs diversity / repetition
!python experiments/exp2_context.py              # prompt length, loss by position
!python experiments/exp3_dynamics.py             # train vs val loss of the main run
!python experiments/exp9_attention.py            # attention heatmaps
!python experiments/exp11_amharic_study.py --reuse_analysis   # Amharic study with the new models
!python experiments/exp12_data_scale.py          # small vs full data on shared unseen text

## 6. Export the models for the demo and download everything

In [ ]:
!python scripts/export_for_demo.py --english checkpoints/small/ckpt_best.pt --amharic checkpoints/amharic/ckpt_best.pt \
    --english_full checkpoints/small_full/ckpt_best.pt --amharic_full checkpoints/amharic_full/ckpt_best.pt
!rm -f trained_outputs.zip && zip -qr trained_outputs.zip app/checkpoints results assets/tokenizers \
    checkpoints/*/log.csv checkpoints/*/summary.json checkpoints/*/loss_curve.png checkpoints/*/meta.json -x "results/runs/*"
!du -h trained_outputs.zip
from google.colab import files
files.download("trained_outputs.zip")

## 7. Back on your laptop
1. Unzip `trained_outputs.zip` into the repository folder (allow overwrite).
2. Check: `streamlit run app/app.py` and `python -m src.generate --ckpt app/checkpoints/english/model.pt --prompt "ROMEO:"`.
3. Commit: `git add -A && git commit -m "Add T4-trained models and results" && git push`.